# RocketPy — Full Pipeline (Flight + Hot-fire + SMT0033)

This notebook replicates the original colab `rocketpy_flight_simulation.ipynb` and adds the full data-processing history:

- retiming onboard time using Ptank correlation
- Pc→Thrust calibration
- flight thrust reconstruction (**no ignition synthesis**)
- Flight vs SMT0033 correlation plots
- export a RASP `.eng` and run RocketPy

**No ignition synthesis** = do not copy HFT4 ignition shape into the flight curve.


In [ ]:
!pip install -q rocketpy numpy matplotlib pandas openpyxl


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from rocketpy import Environment, Rocket, Flight, GenericMotor


## Upload data (if needed)
If your private raw flight logs are not in `data/`, upload them in Colab.


In [ ]:
# from google.colab import files
# uploaded = files.upload()
# print(uploaded.keys())


## Import helpers


In [ ]:
from scripts.flight_thrust_pipeline import (
    load_flight_onboard_semicolon,
    load_telemetry_semicolon,
    load_smt_extracted_csv,
    retime_onboard_by_ptank,
    calibrate_pc_to_thrust_origin,
    calibrate_pc_to_thrust_intercept,
    reconstruct_thrust_from_pc,
    export_rasp_eng,
)


## 1) Load datasets


In [ ]:
ONBOARD_PATH = 'data/telemetry_log_2_cut.csv'   # onboard (semicolon)
TELEM_PATH   = 'data/Pressure-Temperature-Flight-Data_flight2.csv'  # optional telemetry
SMT_PATH     = 'data/SMT0033_extracted.csv'

onboard = load_flight_onboard_semicolon(ONBOARD_PATH)
smt = load_smt_extracted_csv(SMT_PATH)

use_telem = os.path.exists(TELEM_PATH)
if use_telem:
    telem = load_telemetry_semicolon(TELEM_PATH)
    ref = telem
    ref_name = 'telemetry'
else:
    ref = smt
    ref_name = 'smt0033'

print('Onboard N:', len(onboard.t))
print('Reference:', ref_name)


## 2) Retiming onboard timebase using Ptank correlation


In [ ]:
t_real, scale = retime_onboard_by_ptank(onboard, ref, pc_thresh=2.0, n_anchors=20)
print('Retiming scale:', scale)

plt.figure(figsize=(10,5))
plt.plot(t_real, onboard.ptank, 'o-', label='Onboard (retimed)')
plt.plot(ref.t, ref.ptank, 'o-', label=f'Reference ({ref_name})', alpha=0.7)
plt.xlim(-2, max(t_real.max(), ref.t.max()) + 2)
plt.xlabel('Time from ignition [s]')
plt.ylabel('Tank pressure [bar]')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Ptank alignment')
plt.show()


## 3) Pc→Thrust calibration (using SMT0033)


In [ ]:
pc0_smt = np.median(smt.pc[smt.t < 0]) if np.any(smt.t < 0) else np.median(smt.pc[:50])
pcg_smt = np.maximum(smt.pc - pc0_smt, 0)
mask = (smt.t > 0.5) & (pcg_smt > 1.0) & (smt.thrust_N > 5)

cal_origin = calibrate_pc_to_thrust_origin(pcg_smt[mask], smt.thrust_N[mask])
cal_inter  = calibrate_pc_to_thrust_intercept(pcg_smt[mask], smt.thrust_N[mask])

print('Origin fit:', cal_origin)
print('Intercept fit:', cal_inter)


## 4) Reconstruct flight thrust (NO ignition synthesis)


In [ ]:
pc0_f = np.median(onboard.pc[t_real < 0]) if np.any(t_real < 0) else np.median(onboard.pc[:10])
cal = cal_origin
flight_thrust = reconstruct_thrust_from_pc(onboard.pc, cal, pc0=pc0_f)

plt.figure(figsize=(10,5))
plt.plot(t_real, flight_thrust, 'o-')
plt.xlabel('Time from ignition [s]')
plt.ylabel('Thrust [N]')
plt.grid(alpha=0.3)
plt.title('Flight thrust (no ignition synthesis)')
plt.show()


## 5) Correlation: Thrust vs Tank Pressure (Flight vs SMT0033)


In [ ]:
mask_f = (t_real >= 0) & (flight_thrust > 1)
order_f = np.argsort(-onboard.ptank[mask_f])
mask_s = (smt.t >= 0) & (smt.thrust_N > 1)
order_s = np.argsort(-smt.ptank[mask_s])

plt.figure(figsize=(10,6))
plt.plot(smt.ptank[mask_s][order_s], smt.thrust_N[mask_s][order_s], '-', label='SMT0033 (measured)')
plt.plot(onboard.ptank[mask_f][order_f], flight_thrust[mask_f][order_f], '-', label='Flight (reconstructed)')
plt.gca().invert_xaxis()
plt.xlabel('Tank pressure [bar]')
plt.ylabel('Thrust [N]')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Thrust vs Tank Pressure')
plt.show()


## 6) Export `.eng` and run RocketPy


In [ ]:
OUT_ENG = 'Flight_Reconstructed_Retimed_NoIgn.eng'
idx = np.argsort(t_real)
t_sorted = t_real[idx]
f_sorted = flight_thrust[idx]
mask = t_sorted >= 0
te = np.append(t_sorted[mask], t_sorted[mask][-1] + 0.01)
fe = np.append(f_sorted[mask], 0.0)
export_rasp_eng(OUT_ENG, name='FlightRetimedNoIgn', t=te, thrust=fe)
print('Wrote', OUT_ENG)


In [ ]:
# Environment (from rocketpy_flight_simulation.ipynb)
env = Environment(latitude=24.18133, longitude=53.688379, elevation=5)
env.set_date((2026, 2, 13, 12))
env.set_atmospheric_model(
    type='custom_atmosphere',
    wind_u=[(0, 0.07), (10, 0.07), (135, 0.00), (818, -0.45), (1542, -0.62), (3164, -0.51), (5854, 12.69)],
    wind_v=[(0, -4.00), (10, -4.00), (135, -4.19), (818, -1.46), (1542, 1.40), (3164, -0.51), (5854, 4.62)],
    pressure=[(0, 101500), (135, 100000), (818, 92500), (1542, 85000), (3164, 70000), (5854, 50000)],
    temperature=[(0, 302.95), (135, 301.65), (818, 295.25), (1542, 288.75), (3164, 281.35), (5854, 264.25)],
)

burn_time = float(te[-2])
motor = GenericMotor(
    thrust_source=OUT_ENG,
    burn_time=burn_time,
    chamber_radius=0.05,
    chamber_height=1.33,
    chamber_position=1.33/2,
    propellant_initial_mass=2.42,
    nozzle_radius=0.025,
    dry_mass=6.9,
    dry_inertia=(0.5, 0.5, 0.01),
    nozzle_position=0.0,
    center_of_dry_mass_position=1.33/2,
    coordinate_system_orientation='nozzle_to_combustion_chamber',
)

rocket = Rocket(
    radius=0.05,
    mass=3.780,
    inertia=(3.5, 3.5, 0.005),
    power_off_drag='data/poweroff_drag.csv',
    power_on_drag='data/poweron_drag.csv',
    center_of_mass_without_motor=1.869,
    coordinate_system_orientation='tail_to_nose',
)
rocket.add_motor(motor, position=0.0)
rocket.add_nose(length=0.3, kind='ogive', position=2.600)
rocket.add_trapezoidal_fins(n=4, root_chord=0.145, tip_chord=0.065, span=0.08, sweep_length=0.11, cant_angle=0.5, position=0.145)
rocket.add_tail(top_radius=0.05, bottom_radius=0.03, length=0.055, position=0.0)
rocket.set_rail_buttons(upper_button_position=1.80, lower_button_position=0.40, angular_position=88)

def main_trigger(p, h, y):
    return True if y[5] < 0 else False

rocket.add_parachute(
    name='Main',
    cd_s=2.2 * np.pi * (1.8288/2)**2,
    trigger=main_trigger,
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

flight = Flight(rocket=rocket, environment=env, rail_length=7.0, inclination=83.5, heading=90, max_time=600, time_overshoot=True)
print('Apogee AGL [m]:', float(flight.apogee - env.elevation))
